# Demo 00: Run Qwen3-0.6B locally

This notebook verifies the complete local inference path before LoRA fine-tuning:

1. find the downloaded Qwen3-0.6B artifacts;
2. select MPS when it is available, otherwise report an explicit CPU fallback;
3. load the tokenizer and model with Transformers without network access;
4. ask the model one editable question.

Run all cells in order from a clean kernel. The notebook deliberately uses the local model directory and `local_files_only=True`, so it will not download files from Hugging Face.

In [ ]:
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def find_project_root(start_directory: Path) -> Path:
    """Return the repository root containing AGENTS.md and requirements.txt."""
    for candidate in (start_directory, *start_directory.parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'requirements.txt').is_file():
            return candidate
    raise RuntimeError(
        'Could not find the project root. Start JupyterLab from the repository root.'
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
MODEL_DIRECTORY = PROJECT_ROOT / 'artifacts' / 'models' / 'Qwen3-0.6B'
REQUIRED_MODEL_FILES = ('config.json', 'tokenizer.json', 'model.safetensors')

missing_files = [
    filename for filename in REQUIRED_MODEL_FILES if not (MODEL_DIRECTORY / filename).is_file()
]
if missing_files:
    raise FileNotFoundError(
        f'Missing local model artifacts in {MODEL_DIRECTORY}: {missing_files}. '
        'Download the model using the command in the root README before running this demo.'
    )

print(f'Project root: {PROJECT_ROOT}')
print(f'Local model directory: {MODEL_DIRECTORY}')

In [ ]:
DEVICE_REQUEST = 'auto'  # Allowed values: 'auto', 'mps', 'cpu'


def select_device(requested_device: str) -> torch.device:
    """Select a device according to the project's explicit MPS contract."""
    if requested_device not in {'auto', 'mps', 'cpu'}:
        raise ValueError(
            "requested_device must be one of: 'auto', 'mps', or 'cpu'."
        )

    mps_is_built = torch.backends.mps.is_built()
    mps_is_available = torch.backends.mps.is_available()

    if requested_device == 'cpu':
        return torch.device('cpu')
    if requested_device == 'mps':
        if not mps_is_built:
            raise RuntimeError('MPS was requested, but this PyTorch build has no MPS support.')
        if not mps_is_available:
            raise RuntimeError(
                'MPS was requested, but it is unavailable in this runtime. '
                'Check macOS, Apple Silicon support, and the installed PyTorch build.'
            )
        return torch.device('mps')

    if mps_is_available:
        return torch.device('mps')

    print('Warning: MPS is unavailable. Falling back to CPU; inference will be slower.')
    return torch.device('cpu')


MPS_IS_BUILT = torch.backends.mps.is_built()
MPS_IS_AVAILABLE = torch.backends.mps.is_available()
DEVICE = select_device(DEVICE_REQUEST)

print(f'PyTorch version: {torch.__version__}')
print(f'MPS built: {MPS_IS_BUILT}')
print(f'MPS available: {MPS_IS_AVAILABLE}')
print(f'Selected device: {DEVICE}')
print('Inference dtype: torch.float32')

## Load and validate the local model

The first demo uses `torch.float32` as a conservative baseline. It validates the Qwen3 architecture before accepting the model as ready for inference.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIRECTORY,
    local_files_only=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIRECTORY,
    dtype=torch.float32,
    local_files_only=True,
).to(DEVICE)
model.eval()

if model.config.model_type != 'qwen3':
    raise RuntimeError(
        f"Expected a Qwen3 model, but loaded model_type={model.config.model_type!r}."
    )

parameter_device = next(model.parameters()).device
if parameter_device.type != DEVICE.type:
    raise RuntimeError(
        f'Model parameters are on {parameter_device}, expected {DEVICE}. '
        'Stop and inspect the device configuration.'
    )

print('Model loaded successfully.')
print(f'Model type: {model.config.model_type}')
print(f'Model class: {model.__class__.__name__}')
print(f'Model device: {parameter_device}')
print(f'Evaluation mode: {not model.training}')
print(f'Tokenizer class: {tokenizer.__class__.__name__}')

## Ask the model

Change only `USER_PROMPT` and run this cell again to ask another question. Thinking mode is disabled to keep this first demonstration concise.

In [ ]:
USER_PROMPT = 'Explain LoRA fine-tuning in two short sentences for a software developer.'
MAX_NEW_TOKENS = 128

messages = [{'role': 'user', 'content': USER_PROMPT}]
chat_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
model_inputs = tokenizer(chat_text, return_tensors='pt')
model_inputs = {name: tensor.to(DEVICE) for name, tensor in model_inputs.items()}

with torch.inference_mode():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        pad_token_id=tokenizer.eos_token_id,
    )

new_token_ids = generated_ids[0, model_inputs['input_ids'].shape[-1]:]
response = tokenizer.decode(new_token_ids, skip_special_tokens=True).strip()

print(f'Prompt: {USER_PROMPT}')
print('\nResponse:')
print(response)